# Notebook 2: Local and Global PCC

Computes per-subject Pearson correlation matrices from the `rois_aal` (`filt_noglobal`) time series, restricted to the 90 cortical/subcortical AAL regions (header-parsed labels < 9001), in the format `datasets/Dataset.py` expects (patched to use `ASD_*` folder names).

- **Global/Static PCC** (this section): one 90x90 signed correlation matrix per subject from the full time series -> `ASD_ADJ/` or `NC_ADJ/`, key `cropped_matrix`.
- **Local/Dynamic PCC**: separate section, done after this.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import savemat

DATA_DIR = Path("../data/raw")
PHENOTYPIC_PATH = DATA_DIR / "phenotypic_filtered.csv"
ROIS_AAL_DIR = DATA_DIR / "rois_aal"

DATASET_ROOT = Path("../data/GraSTIACL_ABIDE_979/raw")
ASD_ADJ_DIR = DATASET_ROOT / "ASD_ADJ"
NC_ADJ_DIR = DATASET_ROOT / "NC_ADJ"
ASD_ADJ_DIR.mkdir(parents=True, exist_ok=True)
NC_ADJ_DIR.mkdir(parents=True, exist_ok=True)

AAL90_LABEL_CUTOFF = 9001
STD_EPS = 1e-8

## Step 1: Load the filtered phenotypic table (979 subjects)

In [2]:
df = pd.read_csv(PHENOTYPIC_PATH)
print(f"Subjects: {len(df)}")
assert len(df) == 979
df[["FILE_ID", "SITE_ID", "DX_GROUP"]].head()

Subjects: 979


,FILE_ID,SITE_ID,DX_GROUP
0,Pitt_0050003,PITT,1
1,Pitt_0050004,PITT,1
2,Pitt_0050005,PITT,1
3,Pitt_0050006,PITT,1
4,Pitt_0050007,PITT,1


## Step 2: Load rois_aal.1D, parse header for label IDs, keep labels < 9001

Each `.1D` file's first line is a `#`-prefixed, tab-separated header of AAL label IDs (e.g. `#2001	#2002	...`). Parsed directly per file rather than assumed from atlas ordering.

In [3]:
def load_rois_aal90(file_id: str) -> np.ndarray:
    """Load a subject's rois_aal.1D, header-parsed, restricted to AAL-90 labels (< 9001)."""
    path = ROIS_AAL_DIR / f"{file_id}_rois_aal.1D"

    with open(path) as f:
        header_line = f.readline()

    labels = np.array([int(re.sub(r"^#", "", tok)) for tok in header_line.split()])
    keep_mask = labels < AAL90_LABEL_CUTOFF

    ts_full = np.loadtxt(path, skiprows=1)
    ts90 = ts_full[:, keep_mask]

    assert ts90.shape[1] == 90, f"{file_id}: expected 90 AAL-90 columns, got {ts90.shape[1]}"
    return ts90


# sanity check on one subject
_sample = load_rois_aal90(df["FILE_ID"].iloc[0])
print(f"Sample shape: {_sample.shape}")

Sample shape: (196, 90)


## Steps 3-7: Dead-ROI guard, signed global PCC, QC, routing, save

For each subject: guard against dead (near-zero variance) ROIs, compute the signed 90x90 correlation over the full time series, QC it, route by `DX_GROUP` to `ASD_ADJ/` or `NC_ADJ/`, and save as `{FILE_ID}_adj.mat` with key `cropped_matrix`.

In [4]:
qc_records = []
dead_roi_subjects = []

for _, row in df.iterrows():
    file_id = row["FILE_ID"]
    dx_group = int(row["DX_GROUP"])

    ts90 = load_rois_aal90(file_id)
    n_volumes = ts90.shape[0]

    # Step 3: dead-ROI guard (near-zero variance -> NaN correlation)
    roi_std = ts90.std(axis=0)
    n_dead_rois = int((roi_std <= STD_EPS).sum())
    if n_dead_rois > 0:
        dead_roi_subjects.append((file_id, n_dead_rois))

    # Step 4: signed global PCC over the full time series
    W = np.corrcoef(ts90, rowvar=False)

    # Step 5: QC
    is_symmetric = np.allclose(W, W.T, atol=1e-8)
    diag_ok = np.allclose(np.diag(W), 1.0, atol=1e-6)
    is_finite = np.isfinite(W).all()
    in_range = bool((W >= -1.0001).all() and (W <= 1.0001).all())

    qc_records.append(
        {
            "FILE_ID": file_id,
            "DX_GROUP": dx_group,
            "n_volumes": n_volumes,
            "n_dead_rois": n_dead_rois,
            "shape_ok": W.shape == (90, 90),
            "symmetric": is_symmetric,
            "diag_ok": diag_ok,
            "finite": is_finite,
            "in_range": in_range,
        }
    )

    # Step 6: route by DX_GROUP
    out_dir = ASD_ADJ_DIR if dx_group == 1 else NC_ADJ_DIR

    # Step 7: save, signed, key 'cropped_matrix'
    savemat(out_dir / f"{file_id}_adj.mat", {"cropped_matrix": W.astype(np.float64)})

qc_df = pd.DataFrame(qc_records)
print(f"Processed: {len(qc_df)} subjects")
print(f"Subjects with dead ROIs: {len(dead_roi_subjects)}")
if dead_roi_subjects:
    print(dead_roi_subjects[:10])

/users/3171356m/muhammad/GraSTIACL/.venv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/users/3171356m/muhammad/GraSTIACL/.venv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Processed: 979 subjects
Subjects with dead ROIs: 23
[('SDSU_0050184', 4), ('SDSU_0050195', 6), ('SDSU_0050209', 16), ('SDSU_0050216', 11), ('Trinity_0050259', 1), ('CMU_b_0050643', 1), ('CMU_b_0050648', 1), ('CMU_b_0050651', 1), ('CMU_a_0050653', 1), ('CMU_b_0050657', 1)]


## Step 8: Verify output and persist QC (incl. n_volumes for the local-PCC windowing step)

In [5]:
n_asd_files = len(list(ASD_ADJ_DIR.glob("*_adj.mat")))
n_nc_files = len(list(NC_ADJ_DIR.glob("*_adj.mat")))

print(f"ASD_ADJ: {n_asd_files} files (DX_GROUP==1 count: {(df['DX_GROUP'] == 1).sum()})")
print(f"NC_ADJ:  {n_nc_files} files (DX_GROUP==2 count: {(df['DX_GROUP'] == 2).sum()})")

assert n_asd_files == (df["DX_GROUP"] == 1).sum()
assert n_nc_files == (df["DX_GROUP"] == 2).sum()
assert n_asd_files + n_nc_files == len(df)

print()
print("QC flags (all should be True for all subjects):")
print(qc_df[["shape_ok", "symmetric", "diag_ok", "finite", "in_range"]].all())

qc_df.to_csv(DATA_DIR / "global_pcc_qc.csv", index=False)
print(f"\nQC + n_volumes saved to {DATA_DIR / 'global_pcc_qc.csv'}")

ASD_ADJ: 469 files (DX_GROUP==1 count: 469)
NC_ADJ:  510 files (DX_GROUP==2 count: 510)

QC flags (all should be True for all subjects):
shape_ok      True
symmetric    False
diag_ok      False
finite       False
in_range     False
dtype: bool

QC + n_volumes saved to ../data/raw/global_pcc_qc.csv


## Step 9: Drop the 23 dead-ROI subjects, create phenotypic_filtered_v2.csv

ALFF was checked separately and came back 100% clean (979/979 complete, all finite) -- no additional failures to combine. So the final drop list is exactly these 23 dead-ROI subjects. Their `ASD_ADJ`/`NC_ADJ` `.mat` files (which contain NaN) are removed since they're no longer part of the cohort.

In [6]:
dead_roi_ids = qc_df.loc[~qc_df["finite"], "FILE_ID"].tolist()
print(f"Dropping {len(dead_roi_ids)} dead-ROI subjects:")
print(dead_roi_ids)

# remove their stale (NaN-containing) .mat files
for file_id in dead_roi_ids:
    for out_dir in (ASD_ADJ_DIR, NC_ADJ_DIR):
        f = out_dir / f"{file_id}_adj.mat"
        if f.exists():
            f.unlink()

df_v2 = df[~df["FILE_ID"].isin(dead_roi_ids)].reset_index(drop=True)
print(f"\nPhenotypic table: {len(df)} -> {len(df_v2)}")
assert len(df_v2) == len(df) - len(dead_roi_ids)

PHENOTYPIC_V2_PATH = DATA_DIR / "phenotypic_filtered_v2.csv"
df_v2.to_csv(PHENOTYPIC_V2_PATH, index=False)
print(f"Saved: {PHENOTYPIC_V2_PATH}")

# re-verify file counts against the new cohort
n_asd_files = len(list(ASD_ADJ_DIR.glob("*_adj.mat")))
n_nc_files = len(list(NC_ADJ_DIR.glob("*_adj.mat")))
print(f"\nASD_ADJ: {n_asd_files} (expect {(df_v2['DX_GROUP'] == 1).sum()})")
print(f"NC_ADJ:  {n_nc_files} (expect {(df_v2['DX_GROUP'] == 2).sum()})")
assert n_asd_files == (df_v2["DX_GROUP"] == 1).sum()
assert n_nc_files == (df_v2["DX_GROUP"] == 2).sum()
assert n_asd_files + n_nc_files == len(df_v2)
print("\nAll ADJ files now match the 956-subject cohort exactly.")

Dropping 23 dead-ROI subjects:
['SDSU_0050184', 'SDSU_0050195', 'SDSU_0050209', 'SDSU_0050216', 'Trinity_0050259', 'CMU_b_0050643', 'CMU_b_0050648', 'CMU_b_0050651', 'CMU_a_0050653', 'CMU_b_0050657', 'Leuven_2_0050727', 'NYU_0051020', 'NYU_0051118', 'Caltech_0051457', 'Caltech_0051460', 'Caltech_0051464', 'Caltech_0051466', 'Caltech_0051467', 'Caltech_0051469', 'Caltech_0051471', 'Caltech_0051478', 'Caltech_0051483', 'SBL_0051558']

Phenotypic table: 979 -> 956
Saved: ../data/raw/phenotypic_filtered_v2.csv

ASD_ADJ: 455 (expect 455)
NC_ADJ:  501 (expect 501)

All ADJ files now match the 956-subject cohort exactly.


---

## Local/Dynamic PCC

3 non-overlapping windows of 96s each, taken consecutively from the start of each subject's time series (window length in volumes = `round(96 / TR_seconds)`); trailing leftover discarded. Verified beforehand: 0/956 subjects run short (min spare margin = 2 volumes).

Uses the 956-subject cohort, the same `load_rois_aal90` loader, and the same signed-correlation approach as Global PCC -- plus a per-window dead-ROI guard.

### Setup: load the 956-subject cohort, define DW output dirs

In [7]:
df_local = pd.read_csv(PHENOTYPIC_V2_PATH)
print(f"Subjects: {len(df_local)}")
assert len(df_local) == 956

ASD_DW_DIR = DATASET_ROOT / "ASD_DW"
NC_DW_DIR = DATASET_ROOT / "NC_DW"
ASD_DW_DIR.mkdir(parents=True, exist_ok=True)
NC_DW_DIR.mkdir(parents=True, exist_ok=True)

N_WINDOWS = 3
WINDOW_SECONDS = 96

Subjects: 956


### Windowing, per-window dead-ROI guard, signed correlation, QC (no saving yet -- review first)

In [8]:
local_qc_records = []
window_matrices_by_subject = {}  # FILE_ID -> list of 3 (90,90) arrays, kept in memory for the save step

for _, row in df_local.iterrows():
    file_id = row["FILE_ID"]
    dx_group = int(row["DX_GROUP"])
    tr = row["TR_seconds"]

    ts90 = load_rois_aal90(file_id)
    window_volumes = int(round(WINDOW_SECONDS / tr))

    windows = [
        ts90[w * window_volumes : (w + 1) * window_volumes]
        for w in range(N_WINDOWS)
    ]

    window_mats = []
    for w_idx, window in enumerate(windows):
        # per-window dead-ROI guard
        win_std = window.std(axis=0)
        n_dead_rois_window = int((win_std <= STD_EPS).sum())

        Wl = np.corrcoef(window, rowvar=False)
        window_mats.append(Wl)

        is_symmetric = np.allclose(Wl, Wl.T, atol=1e-8)
        diag_ok = np.allclose(np.diag(Wl), 1.0, atol=1e-6)
        is_finite = np.isfinite(Wl).all()
        in_range = bool((Wl >= -1.0001).all() and (Wl <= 1.0001).all())

        local_qc_records.append(
            {
                "FILE_ID": file_id,
                "DX_GROUP": dx_group,
                "window": w_idx,
                "window_volumes": window_volumes,
                "n_dead_rois_window": n_dead_rois_window,
                "shape_ok": Wl.shape == (90, 90),
                "symmetric": is_symmetric,
                "diag_ok": diag_ok,
                "finite": is_finite,
                "in_range": in_range,
            }
        )

    window_matrices_by_subject[file_id] = window_mats

local_qc_df = pd.DataFrame(local_qc_records)
print(f"Processed: {df_local.shape[0]} subjects x {N_WINDOWS} windows = {len(local_qc_df)} window-records")

Processed: 956 subjects x 3 windows = 2868 window-records


In [9]:
subject_any_issue = local_qc_df.groupby("FILE_ID").apply(
    lambda g: (~g["finite"]).any() or (g["n_dead_rois_window"] > 0).any(),
    include_groups=False,
)
flagged_subjects = subject_any_issue[subject_any_issue].index.tolist()

print(f"Subjects with >=1 window issue (dead ROI in a window, or non-finite): {len(flagged_subjects)} / {len(df_local)}")
if flagged_subjects:
    detail = local_qc_df[local_qc_df["FILE_ID"].isin(flagged_subjects)]
    print(detail[["FILE_ID", "window", "n_dead_rois_window", "finite"]].to_string(index=False))

print()
print("QC flags overall (all should be True):")
print(local_qc_df[["shape_ok", "symmetric", "diag_ok", "finite", "in_range"]].all())

local_qc_df.to_csv(DATA_DIR / "local_pcc_qc.csv", index=False)
print(f"\nSaved: {DATA_DIR / 'local_pcc_qc.csv'}")
print("\n(No .mat files written yet -- reviewing this QC table before saving, per plan.)")

Subjects with >=1 window issue (dead ROI in a window, or non-finite): 0 / 956

QC flags overall (all should be True):
shape_ok     True
symmetric    True
diag_ok      True
finite       True
in_range     True
dtype: bool

Saved: ../data/raw/local_pcc_qc.csv

(No .mat files written yet -- reviewing this QC table before saving, per plan.)


### Save: build the (3,1) MATLAB cell array per subject, route by DX_GROUP, save {FILE_ID}_dw.mat

ASD (DX_GROUP==1) -> `ASD_DW/` (Dataset.py assigns y=1); control (DX_GROUP==2) -> `NC_DW/` (y=0). Same routing/label convention as `ASD_ADJ`/`NC_ADJ`.

In [10]:
for _, row in df_local.iterrows():
    file_id = row["FILE_ID"]
    dx_group = int(row["DX_GROUP"])

    window_mats = window_matrices_by_subject[file_id]

    # (3,1) object array -- a MATLAB cell array, one 90x90 matrix per cell,
    # matching Dataset.py's dw_array[j, 0] indexing
    cell_array = np.empty((N_WINDOWS, 1), dtype=object)
    for w_idx, Wl in enumerate(window_mats):
        cell_array[w_idx, 0] = Wl.astype(np.float64)

    out_dir = ASD_DW_DIR if dx_group == 1 else NC_DW_DIR
    savemat(out_dir / f"{file_id}_dw.mat", {"correlation_matrices": cell_array})

n_asd_files = len(list(ASD_DW_DIR.glob("*_dw.mat")))
n_nc_files = len(list(NC_DW_DIR.glob("*_dw.mat")))

print(f"ASD_DW: {n_asd_files} (expect {(df_local['DX_GROUP'] == 1).sum()})")
print(f"NC_DW:  {n_nc_files} (expect {(df_local['DX_GROUP'] == 2).sum()})")
assert n_asd_files == (df_local["DX_GROUP"] == 1).sum()
assert n_nc_files == (df_local["DX_GROUP"] == 2).sum()
assert n_asd_files + n_nc_files == len(df_local)
print("\nAll DW files saved and match the 956-subject cohort exactly.")

ASD_DW: 455 (expect 455)
NC_DW:  501 (expect 501)

All DW files saved and match the 956-subject cohort exactly.
